# EMCCF++ -- split groupe (participant) + CRF-Viterbi (HMR et RTC)

Ce notebook reprend la structure de `HEMC_full.ipynb` (meme repo HEMC :
`hemc.features` / `hemc.models` / `hemc.eval` / `hemc.data`) mais avec une
demarche en deux etapes **par jeu de donnees** :

1. **Etape 1 -- EMCCF++ seul, split aleatoire par echantillon (baseline).**
   Sert uniquement de point de reference rapide, pour verifier que le
   cascade forest EMCCF++ seul (sans aucun post-traitement sequentiel)
   apprend correctement les 4 classes. `sample_train_val_test_split`
   ignore le regroupement par enregistrement : des echantillons consecutifs
   dans le temps (donc tres correles) peuvent se retrouver l'un en train,
   l'autre en test. Consequence directe : un evenement reconstruit a
   partir d'echantillons ainsi disperses n'est plus un evenement
   physiologique reel, donc **le ratio de sur-segmentation
   (`oversegmentation_ratio`) n'est pas rapporte pour ce split** -- il n'a
   de sens que lorsque les sequences temporelles sont intactes (cf.
   Etape 2). Les resultats de cette etape ne sont pas repris dans les
   comparaisons et figures qui suivent : ils servent uniquement de sanity
   check.

2. **Etape 2 -- meme cellule : split groupe (participant/enregistrement) +
   CRF-Viterbi.** C'est **le coeur de ce notebook**. Le split est fait
   **par groupe** (participant) : tous les echantillons d'un meme
   participant vont entierement en train, en validation OU en test --
   jamais reparti entre les trois. C'est le seul split (a) sans fuite
   d'information entre partitions et (b) qui garde des **sequences
   temporellement intactes** par enregistrement dans chaque partition --
   condition indispensable pour un decodage sequentiel (Viterbi a besoin
   de sequences continues pour exploiter durees et transitions
   physiologiques). Tout se passe dans une seule cellule par jeu de
   donnees : split groupe -> cascade forest -> CRF-Viterbi -> metriques
   avant/apres.

   **C'est ce split groupe + CRF qui revele le vrai apport du CRF-Viterbi,
   et c'est la seule comparaison presentee dans les figures et tableaux
   recapitulatifs de ce notebook** (le split aleatoire de l'Etape 1 n'y
   est pas repris). Les figures presentees different par dataset :
   - **HMR** (jeu de reference, quantitatif) : F1 **event-wise** avant/apres
     CRF (IoU >= 0.5) + figure dediee au ratio de **sur-segmentation**
     avant/apres CRF, plus le tableau recapitulatif complet
     (sample-wise + event-wise).
   - **RTC** (enregistrements "terrain", suture robot-assistee) : seule la
     **sur-segmentation avant/apres CRF** est presentee (figure + tableau).
     Aucune metrique de performance du modele (precision/recall/F1) n'est
     montree pour RTC -- ce jeu sert a demontrer l'effet du CRF-Viterbi sur
     la fragmentation des evenements en conditions reelles, pas a comparer
     les performances du classifieur (cf. Etape 2 RTC pour le detail).

Jeux de donnees, dans cet ordre :
- **HMR** (`Data/data_hmr/user_*/eye_*.csv`, format natif du repo HEMC :
  `X_coord`, `Y_coord`, `Confidence`, `Pattern` avec labels F/S/P/B).
- **RTC** (`Data/data_RTCS2/*.csv`, format Pupil Labs : colonnes `gaze x [px]`
  / `gaze y [px]` pour les coordonnees du regard et `classification_Kmeans`
  pour les labels -- mappees vers le meme schema F/S/P/B, voir Etape 0 pour
  le detail du mapping).

3. **Etape 3 (RTC uniquement) -- Stage 2 : fixation vs microsaccade.** RTC
   porte des labels reels de microsaccade (`classification_Kmeans`), donc
   pas besoin de pseudo-labeling comme dans `HEMC_full.ipynb` : un
   classifieur binaire est entraine directement sur la verite terrain, en
   LOSO par participant, avec la **meme methodologie que le notebook de
   reference de l'article** (fenetres a 6 canaux, ResNet1D + Focal Loss +
   Hard Negative Mining progressif) -- voir Etape 3 pour le detail.

## Etape 0 -- imports, chemins, dependances du repo HEMC

In [ ]:
import os
# Fix macOS : evite un crash (SIGABRT / TerminatedWorkerError) des workers
# joblib lances par CascadeForestClassifier (cross_val_predict, n_jobs>1).
# Sur macOS, forker un processus qui a deja charge le runtime Objective-C
# (frameworks Accelerate/Cocoa, souvent via matplotlib ou numpy) declenche
# un abort() de securite lors du fork(). Cette variable desactive ce garde-fou
# -- elle doit etre positionnee avant tout appel parallele (fit() plus bas) ;
# sans effet sur les autres OS.
os.environ.setdefault("OBJC_DISABLE_INITIALIZE_FORK_SAFETY", "YES")

# Fix macOS (2e crash observe malgre OBJC_DISABLE_INITIALIZE_FORK_SAFETY et
# N_JOBS=1) : ce process charge PLUSIEURS copies distinctes de libomp.dylib --
# celle d'Anaconda (via scikit-learn/scipy), celle embarquee par LightGBM
# (lib_lightgbm.dylib) et celle de Homebrew (via XGBoost/libxgboost.dylib).
# Ces runtimes OpenMP concurrents se marchent dessus des qu'ils essaient de
# creer/synchroniser des threads (barrier), d'ou le SIGSEGV observe dans
# libomp.dylib (__kmp_fork_barrier / __kmp_launch_worker) meme sans
# parallelisme joblib. KMP_DUPLICATE_LIB_OK evite l'abort/la collision entre
# copies (fix standard xgboost+lightgbm+sklearn sur macOS) ; OMP_NUM_THREADS=1
# limite en plus la creation de pools de threads OpenMP concurrents. A
# positionner avant tout import de numpy/scikit-learn/xgboost/lightgbm.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Le package `hemc` (features EMCCF++, cascade forest, CRF-Viterbi, metriques)
# vit dans src/ a la racine de CE repo -- lancez ce notebook depuis la racine
# du repo (le dossier contenant `src/hemc/`) pour que Path.cwd() la resolve.
PUPIL_ROOT = Path.cwd()
HEMC_SRC = PUPIL_ROOT / "src"
if not (HEMC_SRC / "hemc").exists():
    raise RuntimeError(
        f"Package hemc introuvable sous {HEMC_SRC}. Lancez ce notebook depuis "
        "la racine du repo (le dossier contenant src/hemc/), ou ajustez PUPIL_ROOT."
    )
sys.path.insert(0, str(HEMC_SRC))

from hemc.data import RecordingMeta, group_train_val_test_split, sample_train_val_test_split, iterate_hmr
from hemc.features import extract_emccfpp_features_df, extract_auxiliary_signal_features_df
from hemc.models import (
    CascadeForestClassifier, SequentialCRFDecoder,
    build_transition_matrix, estimate_mean_durations_ms, estimate_transition_counts, viterbi_decode,
)
from hemc.eval import pointwise_report, event_wise_report
from hemc.utils import set_global_seed

CLASSES_4 = ["F", "S", "P", "B"]  # Fixation, Saccade, (smooth) Pursuit, Blink
set_global_seed(42)

# Donnees et cache : sous-dossiers de ce repo. Data/ et cache/ sont gitignores
# (donnees non redistribuables / artefacts regenerables -- cf. README).
DATA_ROOT = PUPIL_ROOT / "Data"
HMR_ROOT = DATA_ROOT / "data_hmr"
RTC_ROOT = DATA_ROOT / "data_RTCS2"
CACHE_ROOT = PUPIL_ROOT / "cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print(f"HMR_ROOT = {HMR_ROOT} (existe: {HMR_ROOT.exists()})")
print(f"RTC_ROOT = {RTC_ROOT} (existe: {RTC_ROOT.exists()})")

## Configuration

`QUICK = True` charge seulement quelques enregistrements/participants et
reduit la taille du cascade forest, pour valider la structure du notebook en
quelques minutes. Repassez `QUICK` a `False` pour le run de reference a
pleine echelle (tout HMR + tout RTC) -- attendez-vous alors a un temps
d'execution long (le cascade forest EMCCF++ est entraine 2 fois par jeu de
donnees : une fois split aleatoire, une fois split groupe), cf. le
README du repo HEMC, section "Troubleshooting", pour les temps/RAM
attendus a pleine echelle.

In [4]:
QUICK = True  # True = run rapide (structure/plausibilite) ; False = run complet de reference

RANDOM_STATE = 42
N_ESTIMATORS = 15 if QUICK else 150
MAX_LAYERS = 3 if QUICK else 10
# N_JOBS = 1 : le CascadeForestClassifier imbrique deja du parallelisme
# (cross_val_predict(n_jobs=N_JOBS) sur des apprenants RF/ERF/XGB/LGB qui ont
# eux-memes n_jobs=N_JOBS). Avec N_JOBS>1 ce parallelisme imbrique fait forker
# des workers joblib qui relancent XGBoost/LightGBM en multi-thread : sur macOS
# ca segfault de facon intermittente (Fatal Python error: Segmentation fault
# dans xgboost/data.py::_meta_from_numpy, cf. TerminatedWorkerError observe en
# Etape 1). N_JOBS=1 supprime tout parallelisme process/imbrique -- plus lent
# mais fiable ; les tailles reduites de QUICK=True le gardent rapide.
N_JOBS = 1

# LIMIT = nombre d'enregistrements charges (recordings HMR = 1 fichier par oeil,
# fichiers RTC = 1 fichier par participant x session). None = jeu complet.
LIMIT_HMR = 8 if QUICK else None    # 8 recordings = 4 sujets (HMR complet : 13 sujets x 2 yeux = 26 recordings)
LIMIT_RTC = 8 if QUICK else None    # 8 fichiers = 4 participants (RTC complet : 11 participants x 2 sessions = 22 fichiers)

FS_HZ_HMR = 200.0
FS_HZ_RTC = 200.0  # confirme empiriquement : pas temporel median de 0.005s dans timestamp_sec

print(f"QUICK={QUICK} | N_ESTIMATORS={N_ESTIMATORS} | MAX_LAYERS={MAX_LAYERS} | N_JOBS={N_JOBS}")
print(f"LIMIT_HMR={LIMIT_HMR} | LIMIT_RTC={LIMIT_RTC}")


QUICK=True | N_ESTIMATORS=15 | MAX_LAYERS=3 | N_JOBS=1
LIMIT_HMR=8 | LIMIT_RTC=8


## Fonctions communes

Reutilisees a l'identique pour HMR et RTC : chargement + cache des features
EMCCF++, agregation des metriques par evenement, evaluation d'un split, et
la fonction d'Etape 2 qui fait tenir *split groupe + CRF-Viterbi* dans un
seul appel (donc une seule cellule cote "Dataset").

In [ ]:
# --- Chargeur RTC (le repo HEMC ne connait que gazecom/hmr) --------------
# Format Pupil Labs : gaze x/y en pixels, labels dans `classification_Kmeans`.
# Mappe vers le meme schema F/S/P/B que HMR/GazeCom :
#   fixation -> F, saccade -> S, smooth_pursuit -> P, blink -> B
#   microsaccade -> S (petite saccade detectee au sein d'une fixation ;
#     fusionnee avec S faute d'une 5e classe geree par le reste du pipeline)
#   noise -> ecarte (classe non physiologique, marginale : 1 seul fichier concerne)
#
# Signaux auxiliaires RTC-only (absents de HMR, donc features RTC uniquement,
# cf. `get_or_compute_features` plus bas) -- motives par l'article (dossier
# "Hierarchical Classification of Eye Movements...", Sec 2.2.2 Step 0 et
# Sec 3.1.1) :
#   - eyelid_aperture_{mean,min} et pupil_diameter_mean (mm, Pupil Neon) :
#     l'article identifie precisement ces deux signaux comme le cue le plus
#     discriminant pour les blinks (utilises la pour un clustering k-means
#     d'annotation) -- verifie empiriquement ici sur RTC : eyelid aperture
#     moyenne ~12.4mm hors blink vs ~6.1mm pendant un blink (min=0 possible),
#     bien plus separateur que la seule cinematique du regard (qui devient
#     justement peu fiable pendant un blink).
#   - gaze_angular_velocity_deg_s : vitesse angulaire (deg/s), calculee ici a
#     partir de l'azimuth/elevation 3D deja fournis par le pipeline Pupil
#     Neon (donc insensible aux non-linearites pixel<->angle visuel) -- une
#     version plus physiologique de la "velocity" EMCCF++ (Sec 3.1.1), utile
#     en complement pour affiner la frontiere smooth-pursuit / fixation. Le
#     CSV expose deja une colonne `angular_velocity_deg_per_sec` derivee de
#     la meme facon, mais absente d'un des 14 fichiers RTC (Ambl_SUT2) --
#     on la recalcule donc nous-memes depuis azimuth/elevation/timestamp_sec
#     (presents dans les 14 fichiers) plutot que de la lire directement ;
#     verifie empiriquement identique (corr=1.0) a la colonne native la ou
#     elle existe.
RTC_LABEL_MAP = {
    "fixation": "F", "saccade": "S", "microsaccade": "S",
    "smooth_pursuit": "P", "blink": "B",
}

RTC_AUX_RAW_COLUMNS = [
    "pupil diameter left [mm]", "pupil diameter right [mm]",
    "eyelid aperture left [mm]", "eyelid aperture right [mm]",
    "azimuth [deg]", "elevation [deg]", "timestamp_sec",
]


def _gaze_angular_velocity_deg_s(azimuth_deg: np.ndarray, elevation_deg: np.ndarray, timestamp_sec: np.ndarray) -> np.ndarray:
    diffs_t = np.diff(timestamp_sec)
    dt = np.concatenate([[diffs_t[0] if len(diffs_t) else 1.0 / FS_HZ_RTC], diffs_t])
    dt = np.where(dt <= 0, 1.0 / FS_HZ_RTC, dt)  # garde-fou : timestamps non strictement croissants
    d_az = np.diff(azimuth_deg, prepend=azimuth_deg[0])
    d_el = np.diff(elevation_deg, prepend=elevation_deg[0])
    return np.hypot(d_az, d_el) / dt


def load_rtc_recording(path: Path) -> tuple[pd.DataFrame, RecordingMeta]:
    raw = pd.read_csv(path, usecols=["gaze x [px]", "gaze y [px]", "classification_Kmeans", *RTC_AUX_RAW_COLUMNS])
    subject = re.sub(r"_SU[Tt]\d+$", "", path.stem)  # "Mathilde_SUT1" -> "Mathilde"
    recording_id = path.stem
    eyelid_left = raw["eyelid aperture left [mm]"].to_numpy(dtype=float)
    eyelid_right = raw["eyelid aperture right [mm]"].to_numpy(dtype=float)
    angular_velocity = _gaze_angular_velocity_deg_s(
        raw["azimuth [deg]"].to_numpy(dtype=float),
        raw["elevation [deg]"].to_numpy(dtype=float),
        raw["timestamp_sec"].to_numpy(dtype=float),
    )
    df = pd.DataFrame({
        "x": raw["gaze x [px]"].to_numpy(dtype=float),
        "y": raw["gaze y [px]"].to_numpy(dtype=float),
        "label": raw["classification_Kmeans"].map(RTC_LABEL_MAP),  # NaN pour "noise" -> filtre plus bas
        "label_raw": raw["classification_Kmeans"].to_numpy(),  # garde "fixation"/"microsaccade" distincts, pour le Stage 2
        "eyelid_aperture_mean": (eyelid_left + eyelid_right) / 2.0,
        "eyelid_aperture_min": np.minimum(eyelid_left, eyelid_right),  # capte aussi un clignement asymetrique/partiel
        "pupil_diameter_mean": (raw["pupil diameter left [mm]"].to_numpy(dtype=float)
                                 + raw["pupil diameter right [mm]"].to_numpy(dtype=float)) / 2.0,
        "gaze_angular_velocity_deg_s": angular_velocity,
    })
    meta = RecordingMeta(dataset="rtc", recording_id=recording_id, subject=subject, group=subject, fs_hz=FS_HZ_RTC)
    return df, meta


def iterate_rtc(root: Path = RTC_ROOT):
    for csv_path in sorted(root.glob("*.csv")):
        yield load_rtc_recording(csv_path)


# --- Features EMCCF++ (+ auxiliaires RTC blink/SP) + cache disque ---------
# Colonnes auxiliaires RTC-only (cf. `load_rtc_recording` ci-dessus) : si
# presentes dans `df`, on les agrege avec la meme recette multi-echelle
# EMCCF++ (mean/std/max/p25/p90 x 7 fenetres) et on les concatene aux
# features de base -- HMR n'a pas ces colonnes, donc n'est pas affecte
# (feature_cols_hmr/rtc sont calcules independamment par dataset, cf. cellules
# "Dataset 1/2").
RTC_BLINK_AUX_COLUMNS = ["eyelid_aperture_mean", "eyelid_aperture_min", "pupil_diameter_mean"]
RTC_BLINK_AUX_WITH_RATE = {"eyelid_aperture_mean"}  # vitesse de fermeture/ouverture de la paupiere
RTC_SP_AUX_COLUMNS = ["gaze_angular_velocity_deg_s"]
RTC_SP_AUX_WITH_RATE = {"gaze_angular_velocity_deg_s"}  # -> acceleration angulaire


def get_or_compute_features(dataset: str, df: pd.DataFrame, meta: RecordingMeta) -> pd.DataFrame:
    path = CACHE_ROOT / dataset / "emccfpp" / f"{meta.recording_id}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    feats = extract_emccfpp_features_df(df, meta.fs_hz)
    aux_columns = RTC_BLINK_AUX_COLUMNS + RTC_SP_AUX_COLUMNS
    if all(c in df.columns for c in aux_columns):
        aux_blink = extract_auxiliary_signal_features_df(
            df, RTC_BLINK_AUX_COLUMNS, meta.fs_hz, with_rate=RTC_BLINK_AUX_WITH_RATE)
        aux_sp = extract_auxiliary_signal_features_df(
            df, RTC_SP_AUX_COLUMNS, meta.fs_hz, with_rate=RTC_SP_AUX_WITH_RATE)
        feats = pd.concat([feats, aux_blink, aux_sp], axis=1)
    feats["label"] = df["label"].to_numpy()
    feats["group"] = meta.group
    feats["recording_id"] = meta.recording_id
    feats["subject"] = meta.subject
    path.parent.mkdir(parents=True, exist_ok=True)
    feats.to_parquet(path)
    return feats


def build_dataset_table(dataset: str, iterator, limit: int | None = None, verbose: bool = True) -> pd.DataFrame:
    frames = []
    for i, (df, meta) in enumerate(iterator):
        if limit is not None and i >= limit:
            break
        if verbose:
            print(f"  [{dataset}] {meta.recording_id} ({len(df)} echantillons)")
        frames.append(get_or_compute_features(dataset, df, meta))
    table = pd.concat(frames, axis=0, ignore_index=True)
    n_before = len(table)
    table = table[table["label"].isin(CLASSES_4)].reset_index(drop=True)
    n_dropped = n_before - len(table)
    if n_dropped:
        print(f"  {n_dropped} echantillon(s) hors {CLASSES_4} ecarte(s) (ex: 'noise' RTC).")
    return table


# --- Metriques event-wise agregees sur plusieurs enregistrements ----------
def aggregate_event_reports(reports, class_names):
    rows = []
    for c in class_names:
        n_true = sum(r.loc[c, "n_true_events"] for r in reports)
        n_pred = sum(r.loc[c, "n_pred_events"] for r in reports)
        n_matched = sum(round(r.loc[c, "event_precision"] * r.loc[c, "n_pred_events"]) for r in reports)
        precision = n_matched / n_pred if n_pred > 0 else 0.0
        recall = n_matched / n_true if n_true > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        oversegmentation = n_pred / n_true if n_true > 0 else float("nan")
        rows.append({"class": c, "n_true_events": n_true, "n_pred_events": n_pred,
                      "event_precision": precision, "event_recall": recall, "event_f1": f1,
                      "oversegmentation_ratio": oversegmentation})
    return pd.DataFrame(rows).set_index("class")


# --- Etape 1 : evaluation d'un split deja entraine -------------------------
def evaluate_split(clf, table, X, y, test_mask, classes, label, show_oversegmentation=True):
    '''Evalue un split deja entraine, sample-wise et event-wise.

    `show_oversegmentation=False` masque la colonne `oversegmentation_ratio`
    du rapport event-wise affiche : ce ratio n'est interpretable que sur des
    sequences temporellement intactes (split groupe, cf. Etape 2) -- sur un
    split aleatoire par echantillon, les "evenements" reconstruits sont
    artefactuels et ce metrique n'a pas de sens physiologique. Il reste
    calcule (colonne conservee dans le DataFrame retourne, pour un usage
    programmatique) mais n'est simplement pas montre ici.'''
    y_pred = clf.predict(X[test_mask])
    sample_report = pointwise_report(y[test_mask], y_pred, classes)

    test_table = table[test_mask].copy()
    test_table["pred"] = y_pred
    event_reports = [
        event_wise_report(g["pred"].to_numpy(), g["label"].to_numpy(), classes)
        for _, g in test_table.groupby("recording_id", sort=False)
    ]
    event_report = aggregate_event_reports(event_reports, classes)

    print(f"=== {label} -- echantillon par echantillon ===")
    display(sample_report.round(4))
    print(f"\n=== {label} -- par evenement ===")
    if not show_oversegmentation:
        print("(ratio de sur-segmentation non affiche : non interpretable sur un split aleatoire, "
              "les sequences temporelles n'y sont pas intactes -- cf. Etape 2 pour ce metrique.)")
        display(event_report.drop(columns=["oversegmentation_ratio"]).round(4))
    else:
        display(event_report.round(4))
    return sample_report, event_report, y_pred


# --- Etape 2, EN UNE FOIS : split groupe -> cascade forest -> CRF-Viterbi --
def run_group_split_with_crf(table, X, y, feature_cols, classes, fs_hz,
                              n_estimators, max_layers, n_jobs, random_state, context_k=5):
    # split par participant/enregistrement : aucune fuite temporelle, sequences
    # intactes par recording_id -- le seul split compatible avec le CRF-Viterbi
    split_group = group_train_val_test_split(table["group"].tolist(), seed=random_state)
    train_mask = table["group"].isin(split_group.train_groups).to_numpy()
    val_mask = table["group"].isin(split_group.val_groups).to_numpy()
    test_mask = table["group"].isin(split_group.test_groups).to_numpy()
    non_test_mask = ~test_mask
    print(f"Split groupe -- train={train_mask.sum()}, val={val_mask.sum()}, test={test_mask.sum()}")

    clf = CascadeForestClassifier(n_estimators_per_forest=n_estimators, max_layers=max_layers,
                                   random_state=random_state, n_jobs=n_jobs)
    clf.fit(X[train_mask], y[train_mask], X[val_mask], y[val_mask])

    # BUGFIX (cf. HEMC_full.ipynb) : clf.classes_ = np.unique(y_train), trie
    # alphabetiquement -- pas forcement dans l'ordre de `classes`. predict_proba()
    # renvoie ses colonnes dans l'ordre de clf.classes_, donc on reindexe vers
    # `classes` avant tout argmax / alignement avec les durees physiologiques.
    class_order = [list(clf.classes_).index(c) for c in classes]

    # --- CRF-Viterbi : durees/transitions + CRF entraines sur train+val -----
    non_test_label_seqs = [g["label"].to_numpy() for _, g in table[non_test_mask].groupby("recording_id", sort=False)]
    mean_durations_ms = estimate_mean_durations_ms(non_test_label_seqs, classes, fs_hz)
    transition_counts = estimate_transition_counts(non_test_label_seqs, classes)
    transition_matrix = build_transition_matrix(classes, mean_durations_ms, fs_hz, transition_counts)
    print(f"Durees physiologiques moyennes estimees (ms) : {mean_durations_ms}")

    decoder = SequentialCRFDecoder(class_names=classes, context_k=context_k, random_state=random_state)
    context_feats_list, y_context_list = [], []
    for _, g in table[non_test_mask].groupby("recording_id", sort=False):
        proba_seq = clf.predict_proba(g[feature_cols].to_numpy())[:, class_order]
        context_feats_list.append(decoder.build_context_features(proba_seq, mean_durations_ms, fs_hz))
        y_context_list.append(g["label"].to_numpy())
    decoder.fit(np.concatenate(context_feats_list, axis=0), np.concatenate(y_context_list, axis=0))
    print("CRF entraine.")

    # --- evaluation avant / apres CRF sur le test (sequences intactes) ------
    y_pred_before_list, y_pred_after_list, y_true_list = [], [], []
    for _, g in table[test_mask].groupby("recording_id", sort=False):
        proba_seq = clf.predict_proba(g[feature_cols].to_numpy())[:, class_order]
        pred_before = np.array(classes)[proba_seq.argmax(axis=1)]
        feats = decoder.build_context_features(proba_seq, mean_durations_ms, fs_hz)
        smoothed_proba = decoder.predict_proba(feats)
        log_proba = np.log(np.clip(smoothed_proba, 1e-12, 1.0))
        pred_after = np.array(viterbi_decode(log_proba, transition_matrix, classes))

        y_pred_before_list.append(pred_before)
        y_pred_after_list.append(pred_after)
        y_true_list.append(g["label"].to_numpy())

    y_true = np.concatenate(y_true_list)
    report_before = pointwise_report(y_true, np.concatenate(y_pred_before_list), classes)
    report_after = pointwise_report(y_true, np.concatenate(y_pred_after_list), classes)

    event_reports_before = [event_wise_report(p, t, classes) for p, t in zip(y_pred_before_list, y_true_list)]
    event_reports_after = [event_wise_report(p, t, classes) for p, t in zip(y_pred_after_list, y_true_list)]
    agg_before = aggregate_event_reports(event_reports_before, classes)
    agg_after = aggregate_event_reports(event_reports_after, classes)

    # Affichage : uniquement par-evenement ici (le sample-wise reste calcule
    # et retourne -- utilise par compare_sample_vs_event/recap_table plus bas
    # -- mais n'est plus affiche a ce stade pour alleger la sortie).
    print("\n=== AVANT CRF-Viterbi (split groupe) -- par evenement ===")
    display(agg_before.round(4))
    print("\n=== APRES CRF-Viterbi (split groupe) -- par evenement ===")
    display(agg_after.round(4))

    return {
        "clf": clf, "decoder": decoder, "transition_matrix": transition_matrix, "mean_durations_ms": mean_durations_ms,
        "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
        "report_before": report_before, "report_after": report_after,
        "agg_before": agg_before, "agg_after": agg_after,
    }


def compare_sample_vs_event(name, classes, result):
    '''Figure F1 **event-wise** (IoU >= 0.5), avant vs apres CRF-Viterbi, sur
    le split groupe uniquement. Le F1 sample-wise n'est plus trace ici (seul
    le F1 event-wise, plus sensible a la sur-segmentation temporelle, est
    presente en figure) mais reste calcule et retourne pour usage
    programmatique (cf. `recap_table`) et pour le resume global.

    Le split aleatoire de l'Etape 1 n'est volontairement pas repris ici : ses
    metriques event-wise ne sont pas interpretables (sequences non intactes,
    cf. remarque en tete de notebook) -- seule la comparaison avant/apres CRF
    sur le split groupe est presentee comme resultat.'''
    xpos = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(xpos - width / 2, result["agg_before"].loc[classes, "event_f1"], width, label="split groupe -- avant CRF")
    ax.bar(xpos + width / 2, result["agg_after"].loc[classes, "event_f1"], width, label="split groupe -- apres CRF")
    ax.set_xticks(xpos); ax.set_xticklabels(classes)
    ax.set_ylabel("F1 (par evenement, IoU >= 0.5)")
    ax.set_title(f"{name} -- F1 event-wise, avant/apres CRF-Viterbi (split groupe)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    delta_sample = result["report_after"].loc["macro_avg", "f1"] - result["report_before"].loc["macro_avg", "f1"]
    delta_event = result["agg_after"]["event_f1"].mean() - result["agg_before"]["event_f1"].mean()
    print(f"[{name}] Gain macro-F1 apporte par le CRF-Viterbi (split groupe) -- "
          f"sample-wise: {delta_sample:+.4f} | event-wise: {delta_event:+.4f}")
    return delta_sample, delta_event


def plot_oversegmentation(name, classes, result):
    '''Figure dediee a la sur-segmentation temporelle : nombre d'evenements
    predits par evenement reel (`oversegmentation_ratio`), avant vs apres
    CRF-Viterbi, par classe (split groupe uniquement -- seul split avec des
    sequences temporellement intactes, condition necessaire pour que ce ratio
    ait un sens physiologique). Une valeur proche de 1.0 = pas de
    sur-segmentation ; > 1.0 = le modele fragmente les evenements reels en
    plusieurs segments predits.'''
    xpos = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(xpos - width / 2, result["agg_before"].loc[classes, "oversegmentation_ratio"], width,
           label="split groupe -- avant CRF")
    ax.bar(xpos + width / 2, result["agg_after"].loc[classes, "oversegmentation_ratio"], width,
           label="split groupe -- apres CRF")
    ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--", label="1.0 = pas de sur-segmentation")
    ax.set_xticks(xpos); ax.set_xticklabels(classes)
    ax.set_ylabel("Ratio de sur-segmentation\n(evenements predits / evenements reels)")
    ax.set_title(f"{name} -- sur-segmentation, avant/apres CRF-Viterbi (split groupe)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    over_before = result["agg_before"].loc[classes, "oversegmentation_ratio"].mean()
    over_after = result["agg_after"].loc[classes, "oversegmentation_ratio"].mean()
    print(f"[{name}] Sur-segmentation moyenne (macro, split groupe) -- avant CRF: {over_before:.3f} | "
          f"apres CRF: {over_after:.3f} | reduction: {over_before - over_after:+.3f}")
    return over_before, over_after


def _tag(df, tag):
    d = df.copy()
    d.insert(0, "etape", tag)
    return d


def recap_table(classes, result):
    '''Tableau recapitulatif avant/apres CRF-Viterbi, split groupe uniquement
    (le split aleatoire de l'Etape 1 n'est pas repris, cf. remarque en tete
    de notebook et docstring de `compare_sample_vs_event`). Inclut les
    metriques de performance du modele (precision/recall/F1) -- utilise pour
    HMR, le jeu de reference quantitatif de ce notebook (cf. `oversegmentation_table`
    pour la version RTC, sans metriques de performance).'''
    labels_x = classes + ["macro_avg"]
    sample_compare = pd.concat([
        _tag(result["report_before"].loc[labels_x, ["precision", "recall", "f1", "support"]], "split groupe -- avant CRF"),
        _tag(result["report_after"].loc[labels_x, ["precision", "recall", "f1", "support"]], "split groupe -- apres CRF"),
    ])
    event_compare = pd.concat([
        _tag(result["agg_before"][["event_precision", "event_recall", "event_f1", "oversegmentation_ratio"]], "split groupe -- avant CRF"),
        _tag(result["agg_after"][["event_precision", "event_recall", "event_f1", "oversegmentation_ratio"]], "split groupe -- apres CRF"),
    ])
    print("=== Comparatif sample-wise (precision/recall/F1 par classe + macro_avg) ===")
    display(sample_compare.round(4))
    print("\n=== Comparatif event-wise (par classe) -- avant/apres CRF, ratio de sur-segmentation inclus ===")
    display(event_compare.round(4))
    return sample_compare, event_compare


def oversegmentation_table(classes, result):
    '''Tableau recapitulatif de la sur-segmentation avant/apres CRF-Viterbi
    (split groupe), SANS les metriques de performance du modele
    (precision/recall/F1) -- utilise pour RTC, ou seul l'effet du
    CRF-Viterbi sur la sur-segmentation est presente (cf. remarque en tete
    de notebook : RTC sert a demontrer cet effet sur des enregistrements
    "terrain", pas a comparer les performances du modele -- ce dernier
    point est deja couvert par HMR, cf. `recap_table`).'''
    event_compare = pd.concat([
        _tag(result["agg_before"][["n_true_events", "n_pred_events", "oversegmentation_ratio"]], "split groupe -- avant CRF"),
        _tag(result["agg_after"][["n_true_events", "n_pred_events", "oversegmentation_ratio"]], "split groupe -- apres CRF"),
    ])
    print("=== Sur-segmentation par classe -- avant/apres CRF-Viterbi (split groupe) ===")
    display(event_compare.round(4))
    return event_compare


## Dataset 1 -- HMR

In [6]:
table_hmr = build_dataset_table("hmr", iterate_hmr(HMR_ROOT), limit=LIMIT_HMR)
feature_cols_hmr = [c for c in table_hmr.columns if c not in ("label", "group", "recording_id", "subject")]
X_hmr = table_hmr[feature_cols_hmr].to_numpy()
y_hmr = table_hmr["label"].to_numpy()

print(f"\n{len(table_hmr)} echantillons, {len(feature_cols_hmr)} features, "
      f"{table_hmr['recording_id'].nunique()} enregistrements, {table_hmr['group'].nunique()} sujets.")
print(table_hmr["label"].value_counts())


  [hmr] user_1_eye_0 (58255 echantillons)
  [hmr] user_1_eye_1 (58244 echantillons)
  [hmr] user_10_eye_0 (52029 echantillons)
  [hmr] user_10_eye_1 (52029 echantillons)
  [hmr] user_11_eye_0 (53895 echantillons)
  [hmr] user_11_eye_1 (53895 echantillons)
  [hmr] user_12_eye_0 (56189 echantillons)
  [hmr] user_12_eye_1 (56189 echantillons)

440725 echantillons, 189 features, 8 enregistrements, 4 sujets.
label
F    253534
P    111872
B     47772
S     27547
Name: count, dtype: int64


### Etape 1 -- HMR : EMCCF++ seul, split aleatoire par echantillon

`sample_train_val_test_split` ignore le regroupement par enregistrement :
des echantillons consecutifs dans le temps (donc tres correles) peuvent se
retrouver l'un en train, l'autre en test. Les metriques event-wise
ci-dessous sont, de plus, a interpreter avec prudence : un evenement
reconstruit a partir d'echantillons disperses dans le temps n'est plus un
evenement physiologique reel. **Le ratio de sur-segmentation
(`oversegmentation_ratio`) n'est donc pas affiche ici** : il n'est
interpretable que sur des sequences temporellement intactes (cf. Etape 2,
split groupe).


In [7]:
train_mask_r_hmr, val_mask_r_hmr, test_mask_r_hmr = sample_train_val_test_split(len(table_hmr), seed=RANDOM_STATE)
print(f"Split aleatoire -- train={train_mask_r_hmr.sum()}, val={val_mask_r_hmr.sum()}, test={test_mask_r_hmr.sum()}")

clf_random_hmr = CascadeForestClassifier(
    n_estimators_per_forest=N_ESTIMATORS, max_layers=MAX_LAYERS, random_state=RANDOM_STATE, n_jobs=N_JOBS,
)
clf_random_hmr.fit(X_hmr[train_mask_r_hmr], y_hmr[train_mask_r_hmr], X_hmr[val_mask_r_hmr], y_hmr[val_mask_r_hmr])

sample_report_random_hmr, event_report_random_hmr, y_pred_random_hmr = evaluate_split(
    clf_random_hmr, table_hmr, X_hmr, y_hmr, test_mask_r_hmr, CLASSES_4, "HMR -- SPLIT ALEATOIRE PAR ECHANTILLON",
    show_oversegmentation=False,
)


Split aleatoire -- train=352580, val=44072, test=44073


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier w

[CascadeForest] layer 1: val macro-F1 = 0.9274


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with f

[CascadeForest] layer 2: val macro-F1 = 0.9709


...

/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with f

=== HMR -- SPLIT ALEATOIRE PAR ECHANTILLON -- echantillon par echantillon ===


,precision,recall,f1,support
F,0.9886,0.9889,0.9887,25438.0
S,0.9335,0.9417,0.9376,2727.0
P,0.9970,0.9935,0.9952,11169.0
B,0.9762,0.9778,0.9770,4739.0
macro_avg,0.9738,0.9755,0.9746,44073.0



=== HMR -- SPLIT ALEATOIRE PAR ECHANTILLON -- par evenement ===
(ratio de sur-segmentation non affiche : non interpretable sur un split aleatoire, les sequences temporelles n'y sont pas intactes -- cf. Etape 2 pour ce metrique.)


,n_true_events,n_pred_events,event_precision,event_recall,event_f1
class,,,,,
F,1874,1926,0.9559,0.9824,0.9689
S,1070,1109,0.9351,0.9692,0.9518
P,344,352,0.9631,0.9855,0.9741
B,637,658,0.9438,0.9749,0.9591


### Etape 2 -- HMR, meme cellule : split groupe (participant) + CRF-Viterbi

**Pourquoi un split par groupe ?** Ici, le split se fait par **participant**
(`group_train_val_test_split`) : un participant donne se retrouve
entierement en train, en validation OU en test -- jamais reparti entre les
trois. Cela (1) supprime toute fuite d'information entre partitions (deux
echantillons voisins dans le temps, tres correles, ne peuvent plus se
retrouver l'un en train et l'autre en test) et (2) preserve, au sein de
chaque partition, des **sequences temporelles intactes par
enregistrement** -- condition indispensable pour entrainer et evaluer un
decodeur sequentiel.

**Que fait le CRF-Viterbi ?** Le cascade forest EMCCF++ est d'abord
entraine sur train/val comme en Etape 1. Ses probabilites par classe sont
ensuite utilisees pour (a) estimer les **durees physiologiques moyennes**
et les **transitions entre classes** (fixation -> saccade -> ..., a partir
des sequences d'entrainement), et (b) entrainer un CRF sequentiel qui
lisse ces probabilites en tenant compte du contexte temporel. Le decodage
**Viterbi**, combine a la matrice de transition, produit ensuite la
sequence d'etats la plus probable sur le test -- ce qui corrige les
predictions isolees/incoherentes (ex : un seul echantillon de "saccade"
au milieu d'une fixation).

Tout se passe dans une seule cellule via `run_group_split_with_crf` : split
groupe -> cascade forest -> estimation durees/transitions -> entrainement
CRF -> decodage Viterbi -> metriques avant/apres, sample-wise et
event-wise. **Ce sont ces resultats (avant vs apres CRF, split groupe) qui
sont repris dans les comparaisons et le resume global de ce notebook** --
le split aleatoire de l'Etape 1 ne sert que de sanity check et n'est pas
recompare ici.


In [8]:
result_hmr = run_group_split_with_crf(
    table_hmr, X_hmr, y_hmr, feature_cols_hmr, CLASSES_4, FS_HZ_HMR,
    n_estimators=N_ESTIMATORS, max_layers=MAX_LAYERS, n_jobs=N_JOBS, random_state=RANDOM_STATE,
)


Split groupe -- train=220168, val=104058, test=116499


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier w

[CascadeForest] layer 1: val macro-F1 = 0.8530


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier w

[CascadeForest] layer 2: val macro-F1 = 0.8402
[CascadeForest] no improvement after layer 2, stopping at 1 layers
[CascadeForest] final model: 1 layers, best val macro-F1 = 0.8530
Durees physiologiques moyennes estimees (ms) : {'F': 559.4341943419435, 'S': 115.53047404063204, 'P': 1520.6296296296296, 'B': 308.32298136645966}


...

/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature nam


=== AVANT CRF-Viterbi (split groupe) -- par evenement ===


,n_true_events,n_pred_events,event_precision,event_recall,event_f1,oversegmentation_ratio
class,,,,,,
F,516,802,0.5623,0.8740,0.6844,1.5543
S,370,392,0.8520,0.9027,0.8766,1.0595
P,86,361,0.2382,1.0000,0.3848,4.1977
B,94,174,0.3506,0.6489,0.4552,1.8511



=== APRES CRF-Viterbi (split groupe) -- par evenement ===


,n_true_events,n_pred_events,event_precision,event_recall,event_f1,oversegmentation_ratio
class,,,,,,
F,516,556,0.8004,0.8624,0.8302,1.0775
S,370,363,0.9063,0.8892,0.8977,0.9811
P,86,135,0.6074,0.9535,0.7421,1.5698
B,94,85,0.6471,0.5851,0.6145,0.9043


### Comparaison HMR -- F1 event-wise + sur-segmentation, avant vs apres CRF-Viterbi (split groupe)

Comparaison **avant vs apres CRF-Viterbi**, sur le split groupe uniquement
-- le split aleatoire de l'Etape 1 n'est pas repris ici (cf. remarque en
tete de notebook et en Etape 2 : ses metriques event-wise ne sont pas
interpretables sur des sequences non intactes). Deux figures : le F1
event-wise (IoU >= 0.5) par classe, puis le ratio de sur-segmentation par
classe -- suivies du tableau recapitulatif complet (sample-wise + event-wise).

In [ ]:
delta_sample_hmr, delta_event_hmr = compare_sample_vs_event("HMR", CLASSES_4, result_hmr)
over_before_hmr, over_after_hmr = plot_oversegmentation("HMR", CLASSES_4, result_hmr)

In [ ]:
sample_compare_hmr, event_compare_hmr = recap_table(CLASSES_4, result_hmr)

## Dataset 2 -- RTC

In [11]:
table_rtc = build_dataset_table("rtc", iterate_rtc(RTC_ROOT), limit=LIMIT_RTC)
feature_cols_rtc = [c for c in table_rtc.columns if c not in ("label", "group", "recording_id", "subject")]
X_rtc = table_rtc[feature_cols_rtc].to_numpy()
y_rtc = table_rtc["label"].to_numpy()

print(f"\n{len(table_rtc)} echantillons, {len(feature_cols_rtc)} features, "
      f"{table_rtc['recording_id'].nunique()} enregistrements, {table_rtc['group'].nunique()} participants.")
print(table_rtc["label"].value_counts())


  [rtc] Aelie_SUT1_annotation_HH (98905 echantillons)
  [rtc] Aelie_SUT2_annotation_HH (73966 echantillons)
  [rtc] Ambl_SUT1_annotation_HH (123520 echantillons)
  [rtc] Ambl_SUT2_annotation_HH (90021 echantillons)
  [rtc] Enzo_SUT1_annotation_HH (57135 echantillons)
  [rtc] Enzo_SUT2_annotation_HH (72098 echantillons)
  [rtc] Gabrielle_SUT1_annotation_HH (65831 echantillons)
  [rtc] Guillaume_SUT1_annotation_HH (118062 echantillons)
  4293 echantillon(s) hors ['F', 'S', 'P', 'B'] ecarte(s) (ex: 'noise' RTC).

695245 echantillons, 399 features, 8 enregistrements, 8 participants.
label
F    502648
P     84237
B     57147
S     51213
Name: count, dtype: int64


### Etape 1 -- RTC : EMCCF++ seul, split aleatoire par echantillon

Meme remarque qu'en HMR (Etape 1 ci-dessus) : split aleatoire par
echantillon, sequences non intactes. **Le ratio de sur-segmentation
(`oversegmentation_ratio`) n'est donc pas affiche ici** (cf. Etape 2 pour
ce metrique, sur split groupe).


In [12]:
train_mask_r_rtc, val_mask_r_rtc, test_mask_r_rtc = sample_train_val_test_split(len(table_rtc), seed=RANDOM_STATE)
print(f"Split aleatoire -- train={train_mask_r_rtc.sum()}, val={val_mask_r_rtc.sum()}, test={test_mask_r_rtc.sum()}")

clf_random_rtc = CascadeForestClassifier(
    n_estimators_per_forest=N_ESTIMATORS, max_layers=MAX_LAYERS, random_state=RANDOM_STATE, n_jobs=N_JOBS,
)
clf_random_rtc.fit(X_rtc[train_mask_r_rtc], y_rtc[train_mask_r_rtc], X_rtc[val_mask_r_rtc], y_rtc[val_mask_r_rtc])

sample_report_random_rtc, event_report_random_rtc, y_pred_random_rtc = evaluate_split(
    clf_random_rtc, table_rtc, X_rtc, y_rtc, test_mask_r_rtc, CLASSES_4, "RTC -- SPLIT ALEATOIRE PAR ECHANTILLON",
    show_oversegmentation=False,
)


Split aleatoire -- train=556196, val=69524, test=69525


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier w

[CascadeForest] layer 1: val macro-F1 = 0.9125


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier w

[CascadeForest] layer 2: val macro-F1 = 0.9703


...

/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== RTC -- SPLIT ALEATOIRE PAR ECHANTILLON -- echantillon par echantillon ===


,precision,recall,f1,support
F,0.9890,0.9922,0.9906,50281.0
S,0.9299,0.9057,0.9177,5143.0
P,0.9897,0.9854,0.9876,8425.0
B,0.9870,0.9884,0.9877,5676.0
macro_avg,0.9739,0.9679,0.9709,69525.0



=== RTC -- SPLIT ALEATOIRE PAR ECHANTILLON -- par evenement ===
(ratio de sur-segmentation non affiche : non interpretable sur un split aleatoire, les sequences temporelles n'y sont pas intactes -- cf. Etape 2 pour ce metrique.)


,n_true_events,n_pred_events,event_precision,event_recall,event_f1
class,,,,,
F,3531,3578,0.9279,0.9402,0.9340
S,3266,3240,0.9306,0.9231,0.9268
P,725,745,0.9597,0.9862,0.9728
B,462,477,0.9497,0.9805,0.9649


### Etape 2 -- RTC, meme cellule : split groupe (participant) + CRF-Viterbi

Meme methodologie que pour HMR (voir l'explication detaillee en Etape 2 --
Dataset 1 ci-dessus) : split **par participant** (aucune fuite temporelle,
sequences intactes), cascade forest EMCCF++, puis CRF-Viterbi entraine sur
train+val et evalue sur des sequences de test intactes -- avant/apres,
sample-wise et event-wise, dans cette seule cellule via
`run_group_split_with_crf`. Comme pour HMR, **ce sont ces resultats (avant
vs apres CRF, split groupe) qui sont repris dans les comparaisons** ;
le split aleatoire de l'Etape 1 n'est pas recompare.


In [13]:
result_rtc = run_group_split_with_crf(
    table_rtc, X_rtc, y_rtc, feature_cols_rtc, CLASSES_4, FS_HZ_RTC,
    n_estimators=N_ESTIMATORS, max_layers=MAX_LAYERS, n_jobs=N_JOBS, random_state=RANDOM_STATE,
)


Split groupe -- train=525787, val=72098, test=97360


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier w

[CascadeForest] layer 1: val macro-F1 = 0.3663


/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier w

[CascadeForest] layer 2: val macro-F1 = 0.3612
[CascadeForest] no improvement after layer 2, stopping at 1 layers
[CascadeForest] final model: 1 layers, best val macro-F1 = 0.3663
Durees physiologiques moyennes estimees (ms) : {'F': 453.58931201346513, 'S': 46.523346714557356, 'P': 517.9372197309417, 'B': 606.4090909090909}


...

/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/my_env/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b



=== AVANT CRF-Viterbi (split groupe) -- par evenement ===


,n_true_events,n_pred_events,event_precision,event_recall,event_f1,oversegmentation_ratio
class,,,,,,
F,857,927,0.5512,0.5963,0.5729,1.0817
S,836,623,0.8491,0.6328,0.7252,0.7452
P,124,214,0.0654,0.1129,0.0828,1.7258
B,84,102,0.8039,0.9762,0.8817,1.2143



=== APRES CRF-Viterbi (split groupe) -- par evenement ===


,n_true_events,n_pred_events,event_precision,event_recall,event_f1,oversegmentation_ratio
class,,,,,,
F,857,513,0.5673,0.3396,0.4248,0.5986
S,836,538,0.9424,0.6065,0.7380,0.6435
P,124,57,0.4561,0.2097,0.2873,0.4597
B,84,468,0.0171,0.0952,0.0290,5.5714


### Resultats RTC -- sur-segmentation avant vs apres CRF-Viterbi (split groupe)

RTC sert ici a demontrer l'effet du CRF-Viterbi sur la **sur-segmentation**
en conditions "terrain" (suture robot-assistee) -- **aucune metrique de
performance du modele (precision/recall/F1) n'est presentee pour RTC** :
cette comparaison est deja couverte par HMR ci-dessus, le jeu de reference
quantitatif de ce notebook. Seuls la figure et le tableau du ratio de
sur-segmentation (avant/apres CRF, split groupe) suivent.

In [ ]:
over_before_rtc, over_after_rtc = plot_oversegmentation("RTC", CLASSES_4, result_rtc)

In [ ]:
event_compare_rtc = oversegmentation_table(CLASSES_4, result_rtc)

## Etape 3 (RTC uniquement) -- Stage 2 : fixation vs microsaccade

Contrairement a HMR/GazeCom, **RTC porte des labels reels de microsaccade** :
`classification_Kmeans` distingue deja `fixation` de `microsaccade` (pas
besoin de pseudo-labels generes par detecteur comme dans `HEMC_full.ipynb`,
section Stage 2). On entraine donc directement un classifieur binaire
**fixation vs microsaccade** sur la verite terrain RTC, avec la **meme
methodologie que le notebook de reference de l'article**
(`eye_movement_classification_clean2.ipynb`, Stage 2), qui donne les
resultats les plus solides et les plus reproductibles sur ce probleme tres
desequilibre (~1 microsaccade pour 50-60 fixations) :

1. Fenetres glissantes centrees (6 canaux : x, y, vx, vy, vitesse,
   acceleration absolue -- mêmes canaux que le notebook de reference)
   extraites autour des points `fixation`/`microsaccade` -- construits par
   `hemc.stage2.build_channels_monocular` puis `hemc.stage2.extract_windows`.
2. **Augmentation temporelle** des fenetres microsaccade (jitter, decalage,
   etirement, facteur x5) pour compenser leur rarete sans dupliquer les
   memes exemples a l'identique.
3. Un jeu d'entrainement initial construit avec un ratio fixation:MS
   controle (`INITIAL_RATIO`), le reste des fixations formant un **pool**
   dans lequel puiser des negatifs.
4. Un **ResNet1D a blocs residuels dilates** (dilation 1/2/4 -- portage
   PyTorch de l'architecture Keras du notebook de reference), entraine avec
   une **Focal Loss** (`gamma=3.0`, `alpha=0.95`) pour concentrer
   l'apprentissage sur les exemples difficiles/minoritaires.
5. **Hard Negative Mining progressif** : toutes les `REMINE_EVERY` epochs,
   le modele courant re-evalue tout le pool de fixations, et les fenetres
   les plus ambigues (scores eleves) sont re-injectees comme negatifs
   "durs", selon un curriculum en 3 phases (Warm-up : 50% dur / 50% facile
   -- Focus : 70/30 -- Fine-tune : 80/20).
6. Evaluation en **LOSO** (Leave-One-Participant-Out, `hemc.data.loso_folds`),
   avec **seuil de decision optimise** par maximisation du F1 sur la courbe
   precision-rappel (plutot qu'un simple argmax a 0.5), comme dans le
   notebook de reference.

On compare, sur les memes folds LOSO, la meme architecture et la meme
Focal Loss **sans** Hard Negative Mining progressif (dataset initial fixe)
a la version **avec** -- c'est ce levier qui apporte le gain de F1 le plus
important dans le notebook de reference, et c'est lui qu'on cherche a
isoler ici.


In [16]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_curve

from hemc.data import loso_folds
from hemc.stage2 import build_channels_monocular, extract_windows

# --- Config Stage 2, alignee sur la meilleure configuration du notebook de
# reference (`eye_movement_classification_clean2.ipynb`) : fenetre=20,
# stride=1 (un stride dense densifie le nombre de fenetres contenant une
# microsaccade et ameliore le F1, cf. tableau d'ablation du notebook de
# reference). NB : `stride_fraction` est le levier expose par
# `hemc.stage2.extract_windows` pour controler le stride -- on le fixe ici
# pour viser un stride de 1 point ; ajustez STRIDE_TARGET si le comportement
# reel de cette fonction differe (ex : arrondi different de
# round(window_size * stride_fraction)).
WINDOW_SIZE = 20
STRIDE_TARGET = 1
STRIDE_FRACTION = STRIDE_TARGET / WINDOW_SIZE

MAX_LOSO_FOLDS = 4 if QUICK else None   # None = LOSO complet (1 fold par participant RTC, 11 au total)
EPOCHS = 5 if QUICK else 60             # 60 epochs en run complet, comme le notebook de reference
CLASS_NAMES_2 = ["FIXATION", "MS"]
LABEL_TO_IDX_2 = {"FIXATION": 0, "MS": 1}

# --- Hyperparametres Hard Negative Mining progressif + Focal Loss, repris
# tels quels du notebook de reference (Stage 2) ---
AUGMENTATION_FACTOR = 5
INITIAL_RATIO = 5
REMINE_EVERY = 3
FOCAL_GAMMA = 3.0
FOCAL_ALPHA = 0.95
BATCH_SIZE = 32

X_list, y_list, group_list = [], [], []
for i, (df, meta) in enumerate(iterate_rtc(RTC_ROOT)):
    if LIMIT_RTC is not None and i >= LIMIT_RTC:
        break
    label_raw = df["label_raw"].to_numpy()
    ms_region = np.isin(label_raw, ["fixation", "microsaccade"])
    center_labels = np.where(label_raw == "microsaccade", "MS", "FIXATION")
    channels, _ = build_channels_monocular(df["x"].to_numpy(), df["y"].to_numpy(), meta.fs_hz)
    X_w, y_w, _ = extract_windows(channels, center_labels, ms_region, window_size=WINDOW_SIZE, stride_fraction=STRIDE_FRACTION)
    if len(X_w) == 0:
        continue
    X_list.append(X_w)
    y_list.append(y_w)
    group_list.extend([meta.group] * len(X_w))

X_windows_rtc = np.concatenate(X_list)
y_windows_rtc = np.concatenate(y_list)
groups_windows_rtc = np.array(group_list)
y_idx_windows_rtc = np.array([LABEL_TO_IDX_2[v] for v in y_windows_rtc])
print(f"Fenetres extraites : {len(X_windows_rtc)} ({(y_idx_windows_rtc == 1).sum()} MS, "
      f"{(y_idx_windows_rtc == 0).sum()} FIXATION, ratio : "
      f"{(y_idx_windows_rtc == 1).sum() / max((y_idx_windows_rtc == 0).sum(), 1):.3f})")

folds_rtc = loso_folds(groups_windows_rtc)
print(f"{len(folds_rtc)} participant(s) (folds LOSO potentiels)")
if MAX_LOSO_FOLDS is not None and len(folds_rtc) > MAX_LOSO_FOLDS:
    rng = np.random.default_rng(RANDOM_STATE)
    chosen = rng.choice(len(folds_rtc), size=MAX_LOSO_FOLDS, replace=False)
    folds_rtc = [folds_rtc[i] for i in chosen]
print(f"{len(folds_rtc)} fold(s) reellement execute(s)")


Fenetres extraites : 449826 (13270 MS, 436556 FIXATION, ratio : 0.030)
8 participant(s) (folds LOSO potentiels)
4 fold(s) reellement execute(s)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


# --- Focal Loss binaire (sur logits 2 classes), comme dans le notebook de reference ---
class FocalLossBinary(nn.Module):
    def __init__(self, gamma=3.0, alpha=0.95):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)[:, 1]
        targets = targets.float()
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        pt = pt.clamp(min=1e-8, max=1 - 1e-8)
        loss = -alpha_t * (1 - pt).pow(self.gamma) * torch.log(pt)
        return loss.mean()


# --- ResNet1D a blocs residuels dilates -- portage PyTorch de l'architecture
# Keras du notebook de reference (Conv1D 32/64/128, dilation 1/2/4, kernel 5/3/3) ---
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.match = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x if self.match is None else self.match(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class ResNet1DFocal(nn.Module):
    def __init__(self, in_channels, n_classes=2):
        super().__init__()
        self.block1 = ResidualBlock1D(in_channels, 32, kernel_size=5, dilation=1)
        self.block2 = ResidualBlock1D(32, 64, kernel_size=3, dilation=2)
        self.block3 = ResidualBlock1D(64, 128, kernel_size=3, dilation=4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.drop1 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128, 64)
        self.drop2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x).squeeze(-1)
        x = self.drop1(x)
        x = self.relu(self.fc1(x))
        x = self.drop2(x)
        return self.fc2(x)


# --- Augmentation temporelle des fenetres MS (jitter / decalage / etirement),
# reprise telle quelle du notebook de reference (Stage 2) ---
def augment_time_series(X, jitter_std=0.3, shift_max=1, stretch_low=0.95, stretch_high=1.05, p=0.8, seed=42):
    rng = np.random.default_rng(seed)
    X_aug = X.copy()
    n, L, C = X.shape
    for i in range(n):
        if rng.random() < p:
            X_aug[i] += rng.normal(0, jitter_std, size=(L, C))
        if rng.random() < p:
            k = int(rng.integers(-shift_max, shift_max + 1))
            if k > 0:
                X_aug[i, k:, :] = X_aug[i, :-k, :]; X_aug[i, :k, :] = 0
            elif k < 0:
                k = -k
                X_aug[i, :-k, :] = X_aug[i, k:, :]; X_aug[i, -k:, :] = 0
        if rng.random() < p:
            s = float(rng.uniform(stretch_low, stretch_high))
            newL = int(round(L * s))
            idx = np.linspace(0, L - 1, newL)
            Xr = np.stack([np.interp(idx, np.arange(L), X_aug[i, :, c]) for c in range(C)], axis=1)
            if newL >= L:
                X_aug[i] = Xr[:L]
            else:
                pad = np.zeros((L, C), dtype=X.dtype); pad[:newL] = Xr; X_aug[i] = pad
    return X_aug


def prepare_hnm_dataset(X, y_idx, augmentation_factor=5, initial_ratio=5, seed=42):
    '''Augmente les fenetres MS et construit le dataset initial (ratio 1:initial_ratio),
    en conservant un pool complet de fixations pour le mining ulterieur.'''
    rng = np.random.default_rng(seed)
    ms_mask = y_idx == 1
    X_ms, y_ms = X[ms_mask], y_idx[ms_mask]
    X_fix, y_fix = X[~ms_mask], y_idx[~ms_mask]

    X_ms_list, y_ms_list = [X_ms], [y_ms]
    for i in range(augmentation_factor - 1):
        X_ms_list.append(augment_time_series(X_ms, seed=seed + i))
        y_ms_list.append(y_ms.copy())
    X_ms_aug = np.concatenate(X_ms_list)
    y_ms_aug = np.concatenate(y_ms_list)

    n_fix_init = min(len(X_ms_aug) * initial_ratio, len(X_fix))
    fix_idx = rng.choice(len(X_fix), size=n_fix_init, replace=False) if len(X_fix) else np.array([], dtype=int)

    X_init = np.concatenate([X_ms_aug, X_fix[fix_idx]])
    y_init = np.concatenate([y_ms_aug, y_fix[fix_idx]])

    shuffle = rng.permutation(len(X_init))
    return {
        "X_init": X_init[shuffle], "y_init": y_init[shuffle],
        "X_ms_aug": X_ms_aug, "y_ms_aug": y_ms_aug,
        "X_fix_pool": X_fix, "y_fix_pool": y_fix,
    }


def get_mining_strategy(epoch):
    if epoch < 15:
        return {"hard_ratio": 0.5, "phase": "Warm-up"}
    elif epoch < 40:
        return {"hard_ratio": 0.7, "phase": "Focus"}
    return {"hard_ratio": 0.8, "phase": "Fine-tune"}


def train_resnet1d_focal(model, X_init, y_init, X_ms_aug, y_ms_aug, X_fix_pool,
                          epochs, initial_ratio=5, remine_every=3, batch_size=32,
                          use_progressive_hnm=True, verbose=False):
    '''Entraine `model` avec Focal Loss, et (si use_progressive_hnm) re-echantillonne
    periodiquement les negatifs (fixations) selon le curriculum a 3 phases --
    portage PyTorch du callback Keras `ProgressiveHardNegativeMining` du
    notebook de reference (Stage 2).'''
    criterion = FocalLossBinary(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

    X_train_cur, y_train_cur = X_init.copy(), y_init.copy()
    fix_difficulty_scores = np.zeros(len(X_fix_pool)) if len(X_fix_pool) else np.zeros(0)

    for epoch in range(epochs):
        model.train()
        X_t = torch.tensor(X_train_cur, dtype=torch.float32).transpose(1, 2)
        y_t = torch.tensor(y_train_cur, dtype=torch.long)
        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        epoch_loss /= max(len(X_train_cur), 1)
        scheduler.step(epoch_loss)

        if use_progressive_hnm and len(X_fix_pool) and (epoch + 1) % remine_every == 0:
            strategy = get_mining_strategy(epoch)
            model.eval()
            with torch.no_grad():
                X_fix_t = torch.tensor(X_fix_pool, dtype=torch.float32).transpose(1, 2).to(device)
                scores = torch.softmax(model(X_fix_t), dim=1)[:, 1].cpu().numpy()
            alpha = 0.7
            fix_difficulty_scores = alpha * scores + (1 - alpha) * fix_difficulty_scores

            n_total = len(X_ms_aug) * initial_ratio
            n_hard = min(int(n_total * strategy["hard_ratio"]), len(X_fix_pool))
            n_easy = min(n_total - n_hard, len(X_fix_pool))
            hard_idx = np.argsort(fix_difficulty_scores)[-n_hard:] if n_hard else np.array([], dtype=int)
            easy_idx = np.argsort(scores)[:n_easy] if n_easy else np.array([], dtype=int)

            X_train_cur = np.concatenate([X_ms_aug, X_fix_pool[hard_idx], X_fix_pool[easy_idx]])
            y_train_cur = np.concatenate([
                y_ms_aug, np.zeros(len(hard_idx), dtype=int), np.zeros(len(easy_idx), dtype=int),
            ])
            if verbose:
                print(f"    Mining epoch {epoch+1} [{strategy['phase']}] "
                      f"Hard: {len(hard_idx)} | Easy: {len(easy_idx)}")

    return model


def predict_scores(model, X):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).transpose(1, 2).to(device)
        return torch.softmax(model(X_t), dim=1)[:, 1].cpu().numpy()


def best_threshold_f1(y_true, scores):
    '''Seuil qui maximise le F1 sur la courbe precision-rappel, comme dans le
    notebook de reference -- avec un garde-fou : on exclut le dernier point
    (precision=1, recall=0) de `precision_recall_curve`, qui ne correspond a
    aucun seuil reel et provoquerait un IndexError s'il etait selectionne.'''
    P, R, T = precision_recall_curve(y_true, scores)
    if len(T) == 0:
        return 0.5
    F1 = 2 * P * R / (P + R + 1e-12)
    return float(T[np.argmax(F1[:-1])])


Device: cpu


: 

In [ ]:
ablation_reports_rtc = {"sans_hard_negative_mining_progressif": [], "avec_hard_negative_mining_progressif": []}

for fold_i, (train_idx, test_idx, held_out) in enumerate(folds_rtc):
    X_train, y_train = X_windows_rtc[train_idx], y_idx_windows_rtc[train_idx]
    X_test, y_test = X_windows_rtc[test_idx], y_idx_windows_rtc[test_idx]
    if (y_train == 1).sum() == 0 or (y_test == 1).sum() == 0:
        print(f"  fold {fold_i + 1} ({held_out}) : pas de MS des deux cotes, ignore")
        continue

    hnm_data = prepare_hnm_dataset(X_train, y_train, augmentation_factor=AUGMENTATION_FACTOR,
                                    initial_ratio=INITIAL_RATIO, seed=RANDOM_STATE + fold_i)
    X_init, y_init = hnm_data["X_init"], hnm_data["y_init"]
    X_ms_aug, y_ms_aug = hnm_data["X_ms_aug"], hnm_data["y_ms_aug"]
    X_fix_pool = hnm_data["X_fix_pool"]
    in_channels = X_windows_rtc.shape[2]

    # --- avant : Focal Loss + ResNet1D, SANS hard-negative mining progressif
    # (dataset initial fixe, meme ratio, meme architecture, meme loss --
    # seul le mining progressif change entre les deux bras) ---
    model_before = ResNet1DFocal(in_channels=in_channels, n_classes=2).to(device)
    model_before = train_resnet1d_focal(model_before, X_init, y_init, X_ms_aug, y_ms_aug, X_fix_pool,
                                         epochs=EPOCHS, initial_ratio=INITIAL_RATIO, remine_every=REMINE_EVERY,
                                         batch_size=BATCH_SIZE, use_progressive_hnm=False)
    scores_before = predict_scores(model_before, X_test)
    thr_before = best_threshold_f1(y_test, scores_before)
    y_pred_before = np.array(CLASS_NAMES_2)[(scores_before >= thr_before).astype(int)]
    report_before = pointwise_report(np.array(CLASS_NAMES_2)[y_test], y_pred_before, CLASS_NAMES_2)
    ablation_reports_rtc["sans_hard_negative_mining_progressif"].append(report_before)
    del model_before

    # --- apres : + Hard Negative Mining progressif (curriculum Warm-up/Focus/Fine-tune) ---
    model_after = ResNet1DFocal(in_channels=in_channels, n_classes=2).to(device)
    model_after = train_resnet1d_focal(model_after, X_init, y_init, X_ms_aug, y_ms_aug, X_fix_pool,
                                        epochs=EPOCHS, initial_ratio=INITIAL_RATIO, remine_every=REMINE_EVERY,
                                        batch_size=BATCH_SIZE, use_progressive_hnm=True)
    scores_after = predict_scores(model_after, X_test)
    thr_after = best_threshold_f1(y_test, scores_after)
    y_pred_after = np.array(CLASS_NAMES_2)[(scores_after >= thr_after).astype(int)]
    report_after = pointwise_report(np.array(CLASS_NAMES_2)[y_test], y_pred_after, CLASS_NAMES_2)
    ablation_reports_rtc["avec_hard_negative_mining_progressif"].append(report_after)
    del model_after

    if device == "cuda":
        torch.cuda.empty_cache()

    print(f"  fold {fold_i + 1}/{len(folds_rtc)} (participant tenu a l'ecart: {held_out}) : "
          f"F1 MS avant={report_before.loc['MS', 'f1']:.3f} (seuil={thr_before:.3f}), "
          f"apres={report_after.loc['MS', 'f1']:.3f} (seuil={thr_after:.3f})")


In [ ]:
summary_rows = []
for label, reports in ablation_reports_rtc.items():
    if not reports:
        continue
    f1_ms = [r.loc["MS", "f1"] for r in reports]
    precision_ms = [r.loc["MS", "precision"] for r in reports]
    recall_ms = [r.loc["MS", "recall"] for r in reports]
    summary_rows.append({
        "ablation": label, "n_folds": len(reports),
        "precision_MS_mean": np.mean(precision_ms), "recall_MS_mean": np.mean(recall_ms),
        "f1_MS_mean": np.mean(f1_ms), "f1_MS_std": np.std(f1_ms),
    })

if summary_rows:
    stage2_summary_rtc = pd.DataFrame(summary_rows).set_index("ablation")
    print("=== RTC Stage 2 -- F1 microsaccade (vs fixation), avant/apres Hard Negative Mining progressif ===")
    display(stage2_summary_rtc.round(4))

    if len(stage2_summary_rtc) == 2:
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar(stage2_summary_rtc.index, stage2_summary_rtc["f1_MS_mean"], yerr=stage2_summary_rtc["f1_MS_std"], capsize=4)
        ax.set_ylabel("F1 microsaccade (moyenne +/- std sur les folds LOSO)")
        ax.set_title("RTC -- effet du Hard Negative Mining progressif (fixation vs microsaccade)")
        plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
        fig.tight_layout()
        plt.show()
else:
    print("Aucun fold exploitable (pas assez de microsaccades dans le sous-ensemble charge -- "
          "augmentez LIMIT_RTC ou passez QUICK=False).")


## Resume global

Deux tableaux, coherents avec le choix (cf. Etape 2 RTC) de ne presenter des
metriques de **performance du modele** (precision/recall/F1) que pour HMR :

- **Gain de macro-F1** (sample-wise vs event-wise) apporte par le
  CRF-Viterbi sur **HMR** -- jeu de reference quantitatif de ce notebook. Si
  `event-wise > sample-wise` de maniere nette, c'est la confirmation
  recherchee : le CRF corrige surtout la **sur-segmentation temporelle**
  (fragmentation d'evenements physiologiques continus), un effet que les
  metriques point par point sous-estiment.
- **Reduction du ratio de sur-segmentation** apportee par le CRF-Viterbi,
  sur **HMR et RTC** -- la seule comparaison commune aux deux jeux de
  donnees (RTC n'a pas de metriques de performance du modele, cf. Etape 2 RTC).

In [ ]:
summary_f1 = pd.DataFrame([
    {"dataset": "HMR", "gain_macroF1_sample_wise": delta_sample_hmr, "gain_macroF1_event_wise": delta_event_hmr},
]).set_index("dataset")
summary_f1["event_minus_sample"] = summary_f1["gain_macroF1_event_wise"] - summary_f1["gain_macroF1_sample_wise"]
print("=== Gain de macro-F1 apporte par le CRF-Viterbi (HMR uniquement -- RTC n'a pas de "
      "metriques de performance du modele, cf. Etape 2 RTC) ===")
display(summary_f1.round(4))

summary_overseg = pd.DataFrame([
    {"dataset": "HMR", "oversegmentation_avant": over_before_hmr, "oversegmentation_apres": over_after_hmr},
    {"dataset": "RTC", "oversegmentation_avant": over_before_rtc, "oversegmentation_apres": over_after_rtc},
]).set_index("dataset")
summary_overseg["reduction"] = summary_overseg["oversegmentation_avant"] - summary_overseg["oversegmentation_apres"]
print("\n=== Reduction du ratio de sur-segmentation apportee par le CRF-Viterbi (HMR et RTC) ===")
display(summary_overseg.round(4))